# 008 — HyDE: Hypothetical Document Embeddings
**Retrieval Series** | Query expansion through generation

**Problem:** Short queries embed differently from document passages.
`"transformer?"` ≠ embedding of a paragraph explaining transformers.

**HyDE Solution:**
1. Ask the LLM to generate a *hypothetical* answer (no retrieval yet)
2. Embed that hypothetical answer (longer, richer, in-distribution)
3. Use the hypothetical embedding to search — much better signal

**LangGraph** orchestrates the generate → embed → search → answer flow.


In [1]:
import os, warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message=".*LangChain.*")
warnings.filterwarnings('ignore', message=".*allowed_objects.*")
warnings.filterwarnings('ignore', module="langchain.*")
warnings.filterwarnings('ignore', module="langgraph.*")

from dotenv import load_dotenv
load_dotenv(os.path.join(os.getcwd(), '..', '.env'))
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
assert GROQ_API_KEY, "GROQ_API_KEY not found — check ../.env"
print("✓ API key loaded")


✓ API key loaded


In [2]:
import os, time
os.environ["ANONYMIZED_TELEMETRY"] = "False"  # silence ChromaDB telemetry

from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0, max_retries=2)
llm_smart = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, max_retries=2)
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    cache_folder=os.path.join(os.getcwd(), '..', 'datasets', 'models'),
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
print(f"✓ llm       : {llm.model_name}")
print(f"✓ llm_smart : {llm_smart.model_name}")
print(f"✓ embeddings: all-MiniLM-L6-v2 (384-dim, normalized cosine)")


✓ llm       : llama-3.1-8b-instant
✓ llm_smart : llama-3.3-70b-versatile
✓ embeddings: all-MiniLM-L6-v2 (384-dim, normalized cosine)


In [3]:
import os

DS_DIR = os.path.join(os.getcwd(), '..', 'datasets')
os.makedirs(DS_DIR, exist_ok=True)

def load_doc(fname):
    path = os.path.join(DS_DIR, fname)
    if os.path.exists(path):
        return open(path, encoding='utf-8').read()
    raise FileNotFoundError(
        f"Missing: {path}\n"
        "Run  python create_datasets.py  from the project root."
    )

ai_text  = load_doc("ai.txt")
ml_text  = load_doc("machine_learning.txt")
nlp_text = load_doc("nlp.txt")
dl_text  = load_doc("deep_learning.txt")
rag_text = load_doc("rag.txt")
all_text = "\n\n".join([ai_text, ml_text, nlp_text, dl_text, rag_text])

print(f"✓ 5 dataset files loaded")
print(f"  {'File':<20} {'Chars':>8}  {'~Words':>7}")
print(f"  {'-'*40}")
for name, txt in [('ai.txt', ai_text), ('machine_learning.txt', ml_text),
                  ('nlp.txt', nlp_text), ('deep_learning.txt', dl_text),
                  ('rag.txt', rag_text)]:
    print(f"  {name:<20} {len(txt):>8,}  {len(txt.split()):>7,}")
print(f"  {'-'*40}")
print(f"  {'TOTAL':<20} {len(all_text):>8,}  {len(all_text.split()):>7,}")


✓ 5 dataset files loaded
  File                    Chars   ~Words
  ----------------------------------------
  ai.txt                  6,221      943
  machine_learning.txt    5,953      870
  nlp.txt                 5,160      693
  deep_learning.txt       5,114      686
  rag.txt                 5,146      715
  ----------------------------------------
  TOTAL                  27,602    3,907


In [4]:
# ── Build vectorstore from Wikipedia corpus ──────────────────────────────
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS

splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=50)
docs = [Document(page_content=c, metadata={"source": "wiki"})
        for text in [ai_text, ml_text, dl_text]
        for c in splitter.split_text(text[:15000])]

vectorstore = FAISS.from_documents(docs, embeddings)
print(f"✓ Indexed {len(docs)} docs")


✓ Indexed 47 docs


In [5]:
# ── HyDE: generate hypothetical document ─────────────────────────────────
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

hyde_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "Write a short, factual passage (3-4 sentences) that would be the "
        "ideal answer to the following question. Write as if it came from a "
        "Wikipedia article or textbook — do NOT say you don't know."
    )),
    ("human", "{question}"),
])

hyde_chain = hyde_prompt | llm | StrOutputParser()

question = "How does the attention mechanism in transformers work?"
hypothetical_doc = hyde_chain.invoke({"question": question})
print(f"Question: {question}\n")
print(f"Hypothetical document:\n{hypothetical_doc}")


Question: How does the attention mechanism in transformers work?

Hypothetical document:
The attention mechanism in transformers is a key component that enables the model to focus on specific parts of the input sequence when generating output. It works by computing a weighted sum of the input elements, where the weights are determined by the similarity between the input elements and the query vector. This is achieved through the use of a dot product attention function, which calculates the attention weights as the softmax of the dot product between the query vector and the value vectors. The weighted sum of the value vectors is then used as the output of the attention mechanism.


In [6]:
# ── Compare: original query vs HyDE retrieval ────────────────────────────
import numpy as np

# Method A: embed original query
orig_results = vectorstore.similarity_search(question, k=4)

# Method B: embed the hypothetical document
hyde_results = vectorstore.similarity_search(hypothetical_doc, k=4)

print("=== Original query retrieval ===")
for i, d in enumerate(orig_results[:2]):
    print(f"[{i+1}] {d.page_content[:200]}...")

print("\n=== HyDE retrieval ===")
for i, d in enumerate(hyde_results[:2]):
    print(f"[{i+1}] {d.page_content[:200]}...")


=== Original query retrieval ===
[1] Attention and Transformers

The attention mechanism computes a query-key-value system:
Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) * V

Multi-head attention runs multiple attention operations in pa...
[2] The Transformer architecture uses stacked self-attention and feed-forward layers. The encoder
processes the full input simultaneously; the decoder generates output auto-regressively.

Generative Model...

=== HyDE retrieval ===
[1] Attention and Transformers

The attention mechanism computes a query-key-value system:
Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) * V

Multi-head attention runs multiple attention operations in pa...
[2] The Transformer architecture uses stacked self-attention and feed-forward layers. The encoder
processes the full input simultaneously; the decoder generates output auto-regressively.

Generative Model...


In [7]:
# ── LangGraph HyDE pipeline ──────────────────────────────────────────────
from typing import TypedDict
from langgraph.graph import StateGraph, END

class HyDEState(TypedDict):
    question:          str
    hypothetical_doc:  str
    retrieved_docs:    list
    answer:            str

def generate_hypothesis(state: HyDEState) -> HyDEState:
    hyp = hyde_chain.invoke({"question": state["question"]})
    return {**state, "hypothetical_doc": hyp}

def retrieve_with_hyde(state: HyDEState) -> HyDEState:
    docs = vectorstore.similarity_search(state["hypothetical_doc"], k=5)
    return {**state, "retrieved_docs": docs}

def generate_answer(state: HyDEState) -> HyDEState:
    ctx = "\n\n".join(d.page_content for d in state["retrieved_docs"])
    ans_prompt = ChatPromptTemplate.from_template(
        "Answer based on the context.\n\nContext:\n{ctx}\n\nQuestion: {q}\n\nAnswer:"
    )
    ans = (ans_prompt | llm | StrOutputParser()).invoke(
        {"ctx": ctx[:3000], "q": state["question"]}
    )
    return {**state, "answer": ans}

# Build graph
graph = StateGraph(HyDEState)
graph.add_node("generate_hypothesis", generate_hypothesis)
graph.add_node("retrieve",            retrieve_with_hyde)
graph.add_node("gen_answer",          generate_answer)
graph.set_entry_point("generate_hypothesis")
graph.add_edge("generate_hypothesis", "retrieve")
graph.add_edge("retrieve",            "gen_answer")
graph.add_edge("gen_answer",          END)
hyde_app = graph.compile()
print("✓ HyDE LangGraph pipeline compiled")


✓ HyDE LangGraph pipeline compiled


/home/saurabh/miniconda3/lib/python3.13/site-packages/langgraph/checkpoint/base/__init__.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [8]:
# ── Run HyDE pipeline ────────────────────────────────────────────────────
result = hyde_app.invoke({"question": "What is the role of backpropagation in deep learning?"})
print(f"Hypothetical doc (excerpt): {result['hypothetical_doc'][:200]}...")
print(f"\nAnswer: {result['answer']}")


Hypothetical doc (excerpt): Backpropagation is a fundamental algorithm in deep learning that enables the efficient training of artificial neural networks. It is a supervised learning method that calculates the gradient of the lo...

Answer: Backpropagation is the algorithm used to compute gradients in neural networks. It applies the chain rule of calculus to propagate error signals backwards through the network, computing how much each weight contributed to the overall error.


## Key Takeaways
- HyDE consistently improves recall by **5-15%** on typical RAG benchmarks
- The hypothetical document bridges the **query-document semantic gap**
- Best used when queries are short, vague, or domain-specific
- Cost: one extra LLM call per query (cheap with a fast model like llama-3.1-8b-instant)
